---
# Part II – Tweets Emotion Classification using Word Embeddings

In [ ]:
import pandas as pd
import numpy as np
import gensim.downloader as gensim_api
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
twdataset = pd.read_csv('twitter_emotion.csv')
print(twdataset.shape)
print(twdataset.head())

In [ ]:
TWTEXTCOL  = twdataset.columns[0] 
TWLABELCOL = twdataset.columns[-1] 
print(f'Text column : {TWTEXTCOL}')
print(f'Label column: {TWLABELCOL}')
print('\nEmotion distribution:')
print(twdataset[TWLABELCOL].value_counts())

## 1. Tweets Pre-processing – Keras Tokenizer

In [ ]:
MAX_WORDS  = 100_000
MAX_SEQ_LEN = 50         
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(twdataset[TWTEXTCOL].astype(str))
sequences = tokenizer.texts_to_sequences(twdataset[TWTEXTCOL].astype(str))
word_index = tokenizer.word_index
print(f'Vocab size        : {len(word_index)}')
print(f'First 10 entries  : {list(word_index.items())[:10]}')
padded_sequences = pad_sequences(
    sequences,
    maxlen=MAX_SEQ_LEN,
    padding='post',
    truncating='post'
)
print(f'Padded sequences shape: {padded_sequences.shape}')

## 2a. Pre-trained Embeddings – GloVe Twitter 50d

In [ ]:
glove_model = gensim_api.load('glove-twitter-50')
EMBEDDING_DIM_GLOVE = 50
print(f'Vocabulary size in GloVe model: {len(glove_model.key_to_index)}')

In [ ]:
def buildEmbeddingMatrix(wvModel, word_index, max_words, embedding_dimension):
    matrix = np.zeros((max_words + 1, embedding_dimension))
    for word, idx in word_index.items():
        if idx > max_words:
            continue
        try:
            matrix[idx] = wvModel[word]
        except KeyError:
            pass   
    return matrix
glove_embedding_matrix = buildEmbeddingMatrix(
    glove_model, word_index, MAX_WORDS, EMBEDDING_DIM_GLOVE
)
print(f'GloVe embedding matrix shape: {glove_embedding_matrix.shape}')